# Evaluation 2 — Decision Tree Development

This notebook is **Pulmi's primary algorithm contribution**. It justifies Decision Tree classification, establishes the common baseline, evaluates the approved General/City/Resort training scopes, tunes meaningful complexity and class-weight parameters, compares baseline and tuned grouped-CV evidence, generates training-only grouped out-of-fold (OOF) predictions, investigates the City/Resort novelty, and freezes one configuration per scope for later evaluation.

**NO OUTER-TEST MODEL PERFORMANCE IS CALCULATED IN THIS NOTEBOOK.** The outer-test positions are reconstructed only to verify the approved split size. They are never used for fitting, prediction, metrics, or model decisions.

## 1. Imports and shared infrastructure

The existing project modules remain authoritative. Predictor-group hashes are fold-partition helpers—not booking/customer identifiers or model features. Learned preprocessing remains inside every fold-fitted pipeline.

In [1]:
from pathlib import Path
import json
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(root for root in root_candidates if (root / 'src' / 'modeling.py').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import TARGET, prepare_training_table, split_features_target, create_predictor_groups
from src.modeling import (
    RANDOM_STATE, PRIMARY_SCORING, COMMON_SCORING, MODEL_SCOPES, make_inner_cv,
    create_training_scopes, summarize_training_scopes, build_model_pipeline, CV_RESULT_COLUMNS,
)

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hotel_bookings.csv'
CV_OUTPUT_PATH = PROJECT_ROOT / 'reports' / 'eval2' / 'decision_tree_cv_results.csv'
OOF_OUTPUT_PATH = PROJECT_ROOT / 'reports' / 'eval2' / 'decision_tree_oof_results.csv'
RUN_STARTED = time.perf_counter()
captured_warning_messages = []
assert PRIMARY_SCORING == 'f1'
print(f'Project root: {PROJECT_ROOT}')
print(f'Primary scoring metric: {PRIMARY_SCORING}')

Project root: D:\My GitHub Projects\hotel-cancellation
Primary scoring metric: f1


## 2. Reconstruct the approved outer-training population

The approved five-fold `StratifiedGroupKFold` is recreated with shuffling, seed 42, and the first split only. Only training objects are created for development; the test positions exist solely for the required row-count verification.

In [2]:
raw_df = pd.read_csv(RAW_PATH)
training_table = prepare_training_table(raw_df)
X, y = split_features_target(training_table)
predictor_groups = create_predictor_groups(X)

outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_positions, test_positions = next(outer_splitter.split(X, y, groups=predictor_groups))
X_train_outer = X.iloc[train_positions].copy()
y_train_outer = y.iloc[train_positions].copy()
groups_train_outer = predictor_groups.iloc[train_positions].copy()

assert len(X) == 119210
assert len(train_positions) == len(X_train_outer) == 95375
assert len(test_positions) == 23835
assert X_train_outer.shape[1] == 30
assert TARGET not in X_train_outer.columns
assert 'predictor_group' not in X_train_outer.columns
assert not set(groups_train_outer.unique()).intersection(predictor_groups.iloc[test_positions].unique())
display(pd.DataFrame({'measure': ['Modelling rows', 'Outer-training rows', 'Outer-test positions', 'Unencoded predictors'], 'value': [len(X), len(X_train_outer), len(test_positions), X_train_outer.shape[1]]}))

,measure,value
0,Modelling rows,119210
1,Outer-training rows,95375
2,Outer-test positions,23835
3,Unencoded predictors,30


## 3. Approved training scopes

City and Resort are derived from the same fixed outer-training population. Every scope retains the common 30-column schema, including the constant `hotel` predictor in specialist scopes.

In [3]:
scopes = create_training_scopes(X_train_outer, y_train_outer, groups_train_outer)
scope_summary = summarize_training_scopes(scopes)
assert tuple(scopes) == MODEL_SCOPES
assert scope_summary.set_index('scope')['rows'].to_dict() == {'General': 95375, 'City': 63044, 'Resort': 32331}
for scope_name in MODEL_SCOPES:
    assert scopes[scope_name]['X'].shape[1] == 30
    assert 'hotel' in scopes[scope_name]['X'].columns
    assert 'predictor_group' not in scopes[scope_name]['X'].columns
display(scope_summary)

,scope,rows,cancellation_rate,unique_predictor_groups
0,General,95375,0.370779,66831
1,City,63044,0.416915,40872
2,Resort,32331,0.280814,25959


## 4. Decision Tree rationale and baseline

A Decision Tree supports binary classification, learns nonlinear decision boundaries, represents interactions between booking characteristics, and does not require a linear log-odds relationship. Its rule-based splits are interpretable and provide a useful conceptual contrast to linear models. It can learn from the numeric matrix produced by the approved preprocessing pipeline.

Limitations are important: an unrestricted tree can overfit, small data changes can alter its structure, deep trees may memorize training patterns, and an individual tree can be less stable than ensemble methods. Complexity control is therefore central to tuning. No claim is made that Decision Tree will outperform other algorithms.

The common baseline uses Gini impurity, the best splitter, unrestricted depth, the standard `min_samples_split=2` and `min_samples_leaf=1`, no class weighting, and seed 42. It is not manually pruned, uses the estimator's default classification rule, and receives no threshold tuning.

## Baseline Decision Tree Grouped-CV Results

Each result below comes from a complete `preprocessor → classifier` pipeline. Numerical imputation/scaling, categorical missing handling/one-hot vocabulary, and tree structure are learned independently inside each grouped training fold.

In [4]:
BASELINE_PARAMETERS = {
    'criterion': 'gini', 'splitter': 'best', 'max_depth': None,
    'min_samples_split': 2, 'min_samples_leaf': 1,
    'class_weight': None, 'random_state': RANDOM_STATE,
}
BASELINE_JSON = json.dumps(BASELINE_PARAMETERS, sort_keys=True)
METRICS = ('accuracy', 'precision', 'recall', 'f1', 'roc_auc')
cv_records = []

def preserve_warnings(caught, context):
    for item in caught:
        captured_warning_messages.append(f'{context} | {item.category.__name__}: {item.message}')

def make_cv_record(scope_name, stage, summary, *, tuning_method=None, best_parameters=None, observation=''):
    scope = scopes[scope_name]
    record = {column: None for column in CV_RESULT_COLUMNS}
    record.update({
        'algorithm': 'Decision Tree', 'scope': scope_name, 'stage': stage,
        'cv_strategy': 'StratifiedGroupKFold(5)', 'random_state': RANDOM_STATE,
        'parameters': BASELINE_JSON if stage == 'baseline' else json.dumps(summary['parameters'], sort_keys=True),
        'training_rows': len(scope['X']), 'predictor_groups': scope['groups'].nunique(),
        'tuning_method': tuning_method, 'best_parameters': best_parameters, 'observation': observation,
    })
    for metric in METRICS:
        record[f'mean_cv_{metric}'] = float(summary[f'mean_{metric}'])
        record[f'std_cv_{metric}'] = float(summary[f'std_{metric}'])
    return record

baseline_summaries = {}
for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    estimator = DecisionTreeClassifier(**BASELINE_PARAMETERS)
    pipeline = build_model_pipeline(scope['X'], estimator)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        scores = cross_validate(
            pipeline, scope['X'], scope['y'], groups=scope['groups'], cv=make_inner_cv(),
            scoring=COMMON_SCORING, n_jobs=-1, return_train_score=False,
        )
    preserve_warnings(caught, f'baseline {scope_name}')
    summary = {}
    for metric in METRICS:
        summary[f'mean_{metric}'] = scores[f'test_{metric}'].mean()
        summary[f'std_{metric}'] = scores[f'test_{metric}'].std()
    baseline_summaries[scope_name] = summary
    cv_records.append(make_cv_record(scope_name, 'baseline', summary, observation='Unrestricted common baseline evaluated with grouped inner CV.'))

baseline_table = pd.DataFrame([{'scope': s, **baseline_summaries[s]} for s in MODEL_SCOPES])
display(baseline_table)

,scope,mean_accuracy,std_accuracy,mean_precision,std_precision,mean_recall,std_recall,mean_f1,std_f1,mean_roc_auc,std_roc_auc
0,General,0.826531,0.005731,0.769774,0.005516,0.759175,0.013804,0.764398,0.009046,0.814408,0.007634
1,City,0.816348,0.002908,0.782988,0.004947,0.774091,0.006114,0.778491,0.003612,0.813607,0.003237
2,Resort,0.847360,0.005393,0.735536,0.013019,0.713295,0.018930,0.724027,0.010675,0.810313,0.008307


## 5. Grouped GridSearchCV tuning

The exact grid has 2 criteria × 3 depth options × 2 leaf-size options × 2 class-weight options = **24 candidates per scope**, including the baseline. `criterion` changes the split-quality measure; `max_depth` directly controls depth and complexity; `min_samples_leaf` can reduce overly specific terminal rules; and `class_weight` tests whether frequency-based reweighting improves cancellation-oriented metrics.

`splitter='best'`, `min_samples_split=2`, and `random_state=42` remain fixed. The threshold, feature set, splitter, minimum split size, and pruning alpha are not tuned. Each scope uses a fresh estimator, pipeline, and approved grouped splitter; F1 remains the refit metric.

In [5]:
PARAM_GRID = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_leaf': [1, 10],
    'classifier__class_weight': [None, 'balanced'],
}
assert np.prod([len(values) for values in PARAM_GRID.values()]) == 24
best_parameters_by_scope = {}
tuned_summaries = {}
candidate_counts = {}

for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    estimator = DecisionTreeClassifier(**BASELINE_PARAMETERS)
    pipeline = build_model_pipeline(scope['X'], estimator)
    search = GridSearchCV(
        estimator=pipeline, param_grid=PARAM_GRID, scoring=COMMON_SCORING,
        refit=PRIMARY_SCORING, cv=make_inner_cv(), n_jobs=-1, return_train_score=False,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        search.fit(scope['X'], scope['y'], groups=scope['groups'])
    preserve_warnings(caught, f'tuning {scope_name}')
    best_index = search.best_index_
    candidate_counts[scope_name] = len(search.cv_results_['params'])
    assert candidate_counts[scope_name] == 24
    best_parameters_by_scope[scope_name] = dict(search.best_params_)
    summary = {'parameters': {
        'criterion': search.best_params_['classifier__criterion'],
        'splitter': 'best',
        'max_depth': search.best_params_['classifier__max_depth'],
        'min_samples_split': 2,
        'min_samples_leaf': search.best_params_['classifier__min_samples_leaf'],
        'class_weight': search.best_params_['classifier__class_weight'],
        'random_state': RANDOM_STATE,
    }}
    for metric in METRICS:
        summary[f'mean_{metric}'] = search.cv_results_[f'mean_test_{metric}'][best_index]
        summary[f'std_{metric}'] = search.cv_results_[f'std_test_{metric}'][best_index]
    assert np.isclose(summary['mean_f1'], search.best_score_)
    tuned_summaries[scope_name] = summary
    selected_baseline = search.best_params_ == {
        'classifier__criterion': 'gini', 'classifier__max_depth': None,
        'classifier__min_samples_leaf': 1, 'classifier__class_weight': None,
    }
    observation = 'Grid search selected the baseline configuration.' if selected_baseline else 'Grid search selected a different configuration from the baseline.'
    cv_records.append(make_cv_record(
        scope_name, 'tuned', summary, tuning_method='GridSearchCV',
        best_parameters=json.dumps(search.best_params_, sort_keys=True), observation=observation,
    ))
    del search

tuned_table = pd.DataFrame([
    {'scope': s, 'candidates': candidate_counts[s], 'best_parameters': json.dumps(best_parameters_by_scope[s], sort_keys=True),
     **{k: v for k, v in tuned_summaries[s].items() if k != 'parameters'}}
    for s in MODEL_SCOPES
])
display(tuned_table)

,scope,candidates,best_parameters,mean_accuracy,std_accuracy,mean_precision,std_precision,mean_recall,std_recall,mean_f1,std_f1,mean_roc_auc,std_roc_auc
0,General,24,"{""classifier__class_weight"": ""balanced"", ""clas...",0.835971,0.005734,0.745131,0.005927,0.847457,0.009569,0.793004,0.007512,0.916018,0.004224
1,City,24,"{""classifier__class_weight"": null, ""classifier...",0.841711,0.003416,0.839298,0.007787,0.767421,0.011343,0.801659,0.005081,0.917416,0.003683
2,Resort,24,"{""classifier__class_weight"": ""balanced"", ""clas...",0.858557,0.002364,0.690576,0.005566,0.899660,0.019469,0.781224,0.005574,0.924233,0.009797


## 6. Baseline versus tuned validation performance

Differences are descriptive and interpreted separately within each scope. They are not claims of statistical significance or project-wide superiority.

In [6]:
comparison_rows = []
for scope_name in MODEL_SCOPES:
    baseline = baseline_summaries[scope_name]
    tuned = tuned_summaries[scope_name]
    comparison_rows.append({
        'scope': scope_name,
        'baseline_f1': baseline['mean_f1'], 'tuned_f1': tuned['mean_f1'], 'f1_change': tuned['mean_f1'] - baseline['mean_f1'],
        'baseline_recall': baseline['mean_recall'], 'tuned_recall': tuned['mean_recall'], 'recall_change': tuned['mean_recall'] - baseline['mean_recall'],
        'baseline_roc_auc': baseline['mean_roc_auc'], 'tuned_roc_auc': tuned['mean_roc_auc'], 'roc_auc_change': tuned['mean_roc_auc'] - baseline['mean_roc_auc'],
    })
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)

cv_results_df = pd.DataFrame(cv_records, columns=CV_RESULT_COLUMNS)
assert len(cv_results_df) == 6
assert tuple(cv_results_df.columns) == CV_RESULT_COLUMNS
assert set(cv_results_df['stage']) == {'baseline', 'tuned'}
assert set(cv_results_df['scope']) == set(MODEL_SCOPES)
CV_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cv_results_df.to_csv(CV_OUTPUT_PATH, index=False)
print(f'Saved {len(cv_results_df)} standardized CV records to {CV_OUTPUT_PATH}')

,scope,baseline_f1,tuned_f1,f1_change,baseline_recall,tuned_recall,recall_change,baseline_roc_auc,tuned_roc_auc,roc_auc_change
0,General,0.764398,0.793004,0.028606,0.759175,0.847457,0.088283,0.814408,0.916018,0.101610
1,City,0.778491,0.801659,0.023168,0.774091,0.767421,-0.006670,0.813607,0.917416,0.103809
2,Resort,0.724027,0.781224,0.057197,0.713295,0.899660,0.186364,0.810313,0.924233,0.113920


Saved 6 standardized CV records to D:\My GitHub Projects\hotel-cancellation\reports\eval2\decision_tree_cv_results.csv


## 7. Frozen configurations and grouped OOF predictions

Only each search's selected parameter dictionary is retained. Every OOF fold constructs and clones a fresh full pipeline, fits only the fold's training rows, and predicts only validation rows. Probabilities—not hard predictions—are used for ROC-AUC. Assertions enforce zero group overlap and exactly one validation prediction per row.

In [7]:
def grouped_oof_predictions(X_scope, y_scope, groups_scope, selected_parameters, scope_name):
    hard_predictions = np.full(len(X_scope), -1, dtype=int)
    cancellation_probabilities = np.full(len(X_scope), np.nan, dtype=float)
    validation_counts = np.zeros(len(X_scope), dtype=int)
    fold_group_overlaps = []

    for fold_number, (train_idx, valid_idx) in enumerate(
        make_inner_cv().split(X_scope, y_scope, groups=groups_scope), start=1
    ):
        overlap = set(groups_scope.iloc[train_idx]).intersection(groups_scope.iloc[valid_idx])
        fold_group_overlaps.append(len(overlap))
        assert not overlap, f'{scope_name} fold {fold_number} has group overlap'
        estimator = DecisionTreeClassifier(
            criterion=selected_parameters['classifier__criterion'],
            splitter='best',
            max_depth=selected_parameters['classifier__max_depth'],
            min_samples_split=2,
            min_samples_leaf=selected_parameters['classifier__min_samples_leaf'],
            class_weight=selected_parameters['classifier__class_weight'],
            random_state=RANDOM_STATE,
        )
        model = clone(build_model_pipeline(X_scope, estimator))
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            model.fit(X_scope.iloc[train_idx], y_scope.iloc[train_idx])
        preserve_warnings(caught, f'OOF {scope_name} fold {fold_number}')
        hard_predictions[valid_idx] = model.predict(X_scope.iloc[valid_idx])
        cancellation_probabilities[valid_idx] = model.predict_proba(X_scope.iloc[valid_idx])[:, 1]
        validation_counts[valid_idx] += 1

    assert np.all(validation_counts == 1)
    assert np.all(hard_predictions >= 0)
    assert np.isfinite(cancellation_probabilities).all()
    assert max(fold_group_overlaps) == 0
    return hard_predictions, cancellation_probabilities, validation_counts, fold_group_overlaps

oof_predictions = {}
for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    oof_predictions[scope_name] = grouped_oof_predictions(
        scope['X'], scope['y'], scope['groups'], best_parameters_by_scope[scope_name], scope_name,
    )
    print(f'{scope_name}: {len(scope["X"]):,} rows predicted exactly once; fold group overlaps = {oof_predictions[scope_name][3]}')

General: 95,375 rows predicted exactly once; fold group overlaps = [0, 0, 0, 0, 0]


City: 63,044 rows predicted exactly once; fold group overlaps = [0, 0, 0, 0, 0]


Resort: 32,331 rows predicted exactly once; fold group overlaps = [0, 0, 0, 0, 0]


In [8]:
def make_oof_metric_record(scope_name, evaluation_population, y_true, hard_predictions, probabilities):
    return {
        'algorithm': 'Decision Tree', 'scope': scope_name, 'evaluation_population': evaluation_population,
        'stage': 'tuned_oof', 'rows': len(y_true),
        'accuracy': accuracy_score(y_true, hard_predictions),
        'precision': precision_score(y_true, hard_predictions, pos_label=1, zero_division=0),
        'recall': recall_score(y_true, hard_predictions, pos_label=1, zero_division=0),
        'f1': f1_score(y_true, hard_predictions, pos_label=1, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probabilities),
        'parameters': json.dumps(best_parameters_by_scope[scope_name], sort_keys=True),
    }

general = scopes['General']
general_hard, general_probabilities = oof_predictions['General'][:2]
city_mask = general['X']['hotel'].eq('City Hotel').to_numpy()
resort_mask = general['X']['hotel'].eq('Resort Hotel').to_numpy()
oof_records = [
    make_oof_metric_record('General', 'Overall', general['y'], general_hard, general_probabilities),
    make_oof_metric_record('General', 'City', general['y'].iloc[np.flatnonzero(city_mask)], general_hard[city_mask], general_probabilities[city_mask]),
    make_oof_metric_record('General', 'Resort', general['y'].iloc[np.flatnonzero(resort_mask)], general_hard[resort_mask], general_probabilities[resort_mask]),
    make_oof_metric_record('City', 'City', scopes['City']['y'], oof_predictions['City'][0], oof_predictions['City'][1]),
    make_oof_metric_record('Resort', 'Resort', scopes['Resort']['y'], oof_predictions['Resort'][0], oof_predictions['Resort'][1]),
]
oof_results_df = pd.DataFrame(oof_records)
expected_pairs = {('General', 'Overall'), ('General', 'City'), ('General', 'Resort'), ('City', 'City'), ('Resort', 'Resort')}
assert len(oof_results_df) == 5
assert set(zip(oof_results_df['scope'], oof_results_df['evaluation_population'])) == expected_pairs
assert set(oof_results_df['stage']) == {'tuned_oof'}
oof_results_df.to_csv(OOF_OUTPUT_PATH, index=False)
display(oof_results_df)
print(f'Saved {len(oof_results_df)} OOF records to {OOF_OUTPUT_PATH}')

,algorithm,scope,evaluation_population,stage,rows,accuracy,precision,recall,f1,roc_auc,parameters
0,Decision Tree,General,Overall,tuned_oof,95375,0.835963,0.745139,0.847439,0.793004,0.916107,"{""classifier__class_weight"": ""balanced"", ""clas..."
1,Decision Tree,General,City,tuned_oof,63044,0.824694,0.756172,0.855311,0.802692,0.913189,"{""classifier__class_weight"": ""balanced"", ""clas..."
2,Decision Tree,General,Resort,tuned_oof,32331,0.857938,0.713863,0.824650,0.765268,0.915111,"{""classifier__class_weight"": ""balanced"", ""clas..."
3,Decision Tree,City,City,tuned_oof,63044,0.841714,0.839165,0.767425,0.801693,0.917377,"{""classifier__class_weight"": null, ""classifier..."
4,Decision Tree,Resort,Resort,tuned_oof,32331,0.858557,0.690448,0.899659,0.781290,0.924447,"{""classifier__class_weight"": ""balanced"", ""clas..."


Saved 5 OOF records to D:\My GitHub Projects\hotel-cancellation\reports\eval2\decision_tree_oof_results.csv


## 8. Training-only novelty comparison

Specialists are compared only with the General model on the same hotel population. Positive differences mean a higher specialist OOF metric. These are observed grouped training-only validation differences, not proof of final superiority.

In [9]:
novelty_rows = []
for hotel_scope in ('City', 'Resort'):
    general_row = oof_results_df.query("scope == 'General' and evaluation_population == @hotel_scope").iloc[0]
    specialist_row = oof_results_df.query('scope == @hotel_scope and evaluation_population == @hotel_scope').iloc[0]
    novelty_rows.append({
        'population': hotel_scope,
        'general_f1': general_row['f1'], 'specialist_f1': specialist_row['f1'], 'specialist_minus_general_f1': specialist_row['f1'] - general_row['f1'],
        'general_recall': general_row['recall'], 'specialist_recall': specialist_row['recall'], 'specialist_minus_general_recall': specialist_row['recall'] - general_row['recall'],
        'general_roc_auc': general_row['roc_auc'], 'specialist_roc_auc': specialist_row['roc_auc'], 'specialist_minus_general_roc_auc': specialist_row['roc_auc'] - general_row['roc_auc'],
    })
novelty_comparison = pd.DataFrame(novelty_rows)
display(novelty_comparison)

,population,general_f1,specialist_f1,specialist_minus_general_f1,general_recall,specialist_recall,specialist_minus_general_recall,general_roc_auc,specialist_roc_auc,specialist_minus_general_roc_auc
0,City,0.802692,0.801693,-0.000999,0.855311,0.767425,-0.087886,0.913189,0.917377,0.004188
1,Resort,0.765268,0.781290,0.016022,0.824650,0.899659,0.075008,0.915111,0.924447,0.009336


## Decision Tree Development Findings

In [10]:
def complexity_interpretation(scope_name):
    params = best_parameters_by_scope[scope_name]
    depth = params['classifier__max_depth']
    leaf = params['classifier__min_samples_leaf']
    weight = params['classifier__class_weight']
    depth_text = (
        f'a finite maximum depth of {depth}, constraining depth relative to the unrestricted baseline'
        if depth is not None else 'the unrestricted-depth option from the tested grid'
    )
    leaf_text = (
        'larger terminal leaves of at least 10 records, reducing very specific splits'
        if leaf == 10 else 'the minimum leaf size of 1 from the tested grid'
    )
    weight_text = (
        'balanced class weighting, giving greater relative influence according to class frequency'
        if weight == 'balanced' else 'no class reweighting'
    )
    return f"{scope_name}: criterion={params['classifier__criterion']}; selected {depth_text}; selected {leaf_text}; selected {weight_text}."

finding_lines = []
for scope_name in MODEL_SCOPES:
    finding_lines.append(
        f"{scope_name} baseline F1={baseline_summaries[scope_name]['mean_f1']:.4f}; "
        f"tuned F1={tuned_summaries[scope_name]['mean_f1']:.4f}; "
        f"change={tuned_summaries[scope_name]['mean_f1'] - baseline_summaries[scope_name]['mean_f1']:+.4f}. "
        + complexity_interpretation(scope_name)
    )
for row in novelty_rows:
    finding_lines.append(
        f"{row['population']} same-population OOF specialist-minus-General differences: "
        f"F1={row['specialist_minus_general_f1']:+.4f}, "
        f"Recall={row['specialist_minus_general_recall']:+.4f}, "
        f"ROC-AUC={row['specialist_minus_general_roc_auc']:+.4f}."
    )
finding_lines.extend([
    'Decision Trees offer nonlinear, interaction-aware, rule-based modelling, but individual trees can overfit and be structurally unstable under data changes.',
    'No other algorithm, threshold adjustment, or outer-test result is used in this notebook.',
    'The Decision Tree configurations are now frozen for later final outer-holdout evaluation and cross-algorithm comparison.',
])
display(Markdown('\n'.join(f'- {line}' for line in finding_lines)))

- General baseline F1=0.7644; tuned F1=0.7930; change=+0.0286. General: criterion=entropy; selected a finite maximum depth of 20, constraining depth relative to the unrestricted baseline; selected larger terminal leaves of at least 10 records, reducing very specific splits; selected balanced class weighting, giving greater relative influence according to class frequency.
- City baseline F1=0.7785; tuned F1=0.8017; change=+0.0232. City: criterion=gini; selected a finite maximum depth of 10, constraining depth relative to the unrestricted baseline; selected the minimum leaf size of 1 from the tested grid; selected no class reweighting.
- Resort baseline F1=0.7240; tuned F1=0.7812; change=+0.0572. Resort: criterion=gini; selected a finite maximum depth of 10, constraining depth relative to the unrestricted baseline; selected the minimum leaf size of 1 from the tested grid; selected balanced class weighting, giving greater relative influence according to class frequency.
- City same-population OOF specialist-minus-General differences: F1=-0.0010, Recall=-0.0879, ROC-AUC=+0.0042.
- Resort same-population OOF specialist-minus-General differences: F1=+0.0160, Recall=+0.0750, ROC-AUC=+0.0093.
- Decision Trees offer nonlinear, interaction-aware, rule-based modelling, but individual trees can overfit and be structurally unstable under data changes.
- No other algorithm, threshold adjustment, or outer-test result is used in this notebook.
- The Decision Tree configurations are now frozen for later final outer-holdout evaluation and cross-algorithm comparison.

In [11]:
elapsed_minutes = (time.perf_counter() - RUN_STARTED) / 60
print(f'Notebook modelling runtime: {elapsed_minutes:.2f} minutes')
if captured_warning_messages:
    print('Captured Decision Tree warnings require review:')
    for message in captured_warning_messages:
        print('-', message)
else:
    print('No Decision Tree warnings occurred.')
print('Verification complete: 119,210 modelling rows; 95,375 outer-training rows; 23,835 outer-test positions.')
print('All three searches evaluated exactly 24 candidates; all OOF folds had zero group overlap.')
print('No outer-test model prediction or metric was calculated; no threshold tuning was performed.')

Notebook modelling runtime: 7.20 minutes
No Decision Tree warnings occurred.
Verification complete: 119,210 modelling rows; 95,375 outer-training rows; 23,835 outer-test positions.
All three searches evaluated exactly 24 candidates; all OOF folds had zero group overlap.
No outer-test model prediction or metric was calculated; no threshold tuning was performed.
